In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

data = pd.read_csv('voice.csv')
data.head()
data.info()
data['label'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3168 entries, 0 to 3167
Data columns (total 21 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   meanfreq  3168 non-null   float64
 1   sd        3168 non-null   float64
 2   median    3168 non-null   float64
 3   Q25       3168 non-null   float64
 4   Q75       3168 non-null   float64
 5   IQR       3168 non-null   float64
 6   skew      3168 non-null   float64
 7   kurt      3168 non-null   float64
 8   sp.ent    3168 non-null   float64
 9   sfm       3168 non-null   float64
 10  mode      3168 non-null   float64
 11  centroid  3168 non-null   float64
 12  meanfun   3168 non-null   float64
 13  minfun    3168 non-null   float64
 14  maxfun    3168 non-null   float64
 15  meandom   3168 non-null   float64
 16  mindom    3168 non-null   float64
 17  maxdom    3168 non-null   float64
 18  dfrange   3168 non-null   float64
 19  modindx   3168 non-null   float64
 20  label     3168 non-null   obje

,count
label,
male,1584
female,1584


In [ ]:
# memisahkan fitur dan label
X = data.drop('label', axis=1)
y = data['label']

In [ ]:
# membagi data train dan test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
# melakukan scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# coba kNN dengan k = 5
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

In [ ]:
# evaluasi model
akurasi = accuracy_score(y_test, y_pred)

print('Akurasi:', akurasi)
print()
print(classification_report(y_test, y_pred))

Akurasi: 0.9763406940063092

              precision    recall  f1-score   support

      female       0.99      0.97      0.98       317
        male       0.97      0.99      0.98       317

    accuracy                           0.98       634
   macro avg       0.98      0.98      0.98       634
weighted avg       0.98      0.98      0.98       634



In [ ]:
# 2. percobaan fitur yang paling optimal
data_korelasi = data.copy()

data_korelasi['label'] = data_korelasi['label'].map({
    'male': 0,
    'female': 1
})

In [ ]:
# lihat korelasi label dengan tiap column
korelasi = data_korelasi.corr()['label'].abs().sort_values(
    ascending=False
)

print(korelasi)

label       1.000000
meanfun     0.833921
IQR         0.618916
Q25         0.511455
sp.ent      0.490552
sd          0.479539
sfm         0.357499
meanfreq    0.337415
centroid    0.337415
median      0.283919
maxdom      0.195657
mindom      0.194974
dfrange     0.192213
meandom     0.191067
mode        0.171775
maxfun      0.166461
minfun      0.136692
kurt        0.087195
Q75         0.066906
skew        0.036627
modindx     0.030801
Name: label, dtype: float64


In [ ]:
# coba menambahkan fitur satu per satu berdasarkan urutan korelasi

percobaan_fitur = [
    ['meanfun'],

    ['meanfun', 'IQR'],

    ['meanfun', 'IQR', 'Q25'],

    ['meanfun', 'IQR', 'Q25', 'sp.ent'],

    ['meanfun', 'IQR', 'Q25', 'sp.ent', 'sd'],

    ['meanfun', 'IQR', 'Q25', 'sp.ent', 'sd', 'sfm'],

    ['meanfun', 'IQR', 'Q25', 'sp.ent', 'sd',
     'sfm', 'centroid'],

    ['meanfun', 'IQR', 'Q25', 'sp.ent', 'sd',
     'sfm', 'centroid', 'meanfreq'],

    ['meanfun', 'IQR', 'Q25', 'sp.ent', 'sd',
     'sfm', 'centroid', 'meanfreq', 'median']
]

In [ ]:
# uji dengan k = 5

for fitur in percobaan_fitur:

    X = data[fitur]
    y = data['label']

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    knn = KNeighborsClassifier(n_neighbors=5)

    knn.fit(X_train, y_train)

    y_pred = knn.predict(X_test)

    akurasi = accuracy_score(y_test, y_pred)

    print(
        'Jumlah fitur:', len(fitur),
        '| Akurasi:', round(akurasi * 100, 2), '%'
    )

Jumlah fitur: 1 | Akurasi: 95.27 %
Jumlah fitur: 2 | Akurasi: 97.79 %
Jumlah fitur: 3 | Akurasi: 97.79 %
Jumlah fitur: 4 | Akurasi: 97.63 %
Jumlah fitur: 5 | Akurasi: 98.11 %
Jumlah fitur: 6 | Akurasi: 98.58 %
Jumlah fitur: 7 | Akurasi: 98.26 %
Jumlah fitur: 8 | Akurasi: 98.11 %
Jumlah fitur: 9 | Akurasi: 98.58 %


Dari percobaan tersebut, saya memilih 6 fitur karena sudah mendapatkan akurasi tertinggi tanpa perlu menggunakan fitur yang lebih banyak.

Dari percobaan tersebut, saya memilih 6 fitur karena sudah mendapatkan akurasi tertinggi tanpa perlu menggunakan fitur yang lebih banyak.

In [ ]:
# 3. mencari nilai k terbaik

# menggunakan 6 fitur dari nomor sebelumnya

fitur_terbaik = [
    'meanfun',
    'IQR',
    'Q25',
    'sp.ent',
    'sd',
    'sfm'
]

X = data[fitur_terbaik]
y = data['label']

In [ ]:
# split data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# scaling

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# mencoba nilai k dari 1 - 20

k_values = range(1, 21)
akurasi = []

for k in k_values:

    knn = KNeighborsClassifier(n_neighbors=k)

    knn.fit(X_train, y_train)

    y_pred = knn.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    akurasi.append(acc)

    print(
        'k =', k,
        '| Akurasi =',
        round(acc * 100, 2), '%'
    )

k = 1 | Akurasi = 98.58 %
k = 2 | Akurasi = 98.42 %
k = 3 | Akurasi = 98.74 %
k = 4 | Akurasi = 98.58 %
k = 5 | Akurasi = 98.58 %
k = 6 | Akurasi = 98.58 %
k = 7 | Akurasi = 98.11 %
k = 8 | Akurasi = 98.26 %
k = 9 | Akurasi = 97.79 %
k = 10 | Akurasi = 98.11 %
k = 11 | Akurasi = 97.95 %
k = 12 | Akurasi = 97.79 %
k = 13 | Akurasi = 97.79 %
k = 14 | Akurasi = 97.63 %
k = 15 | Akurasi = 97.63 %
k = 16 | Akurasi = 97.95 %
k = 17 | Akurasi = 97.79 %
k = 18 | Akurasi = 97.63 %
k = 19 | Akurasi = 97.63 %
k = 20 | Akurasi = 97.63 %


Dari grafik terlihat bahwa nilai tertinggi berada pada:
k = 3
dengan akurasi:
98,74%